# Consultas SQL · cuaderno del proyecto
**Estud-IA · Análisis de Datos Básico**

Corre las celdas en orden con el botón ▶ (o Shift + Enter). Si Colab muestra una advertencia porque el cuaderno viene de GitHub, elige **«Ejecutar de todos modos»**.

1. La celda 1 prepara todo. No hay que cambiar nada.
2. La celda 2 carga tu muestra.
3. La celda 3 cuenta las filas: tu primera **cifra de control**.
4. Desde la sesión 3: tus dos consultas, más abajo.

In [1]:
# Celda 1 · Preparar (no cambies nada)
try:
    import duckdb
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
    import duckdb
import re
import pandas as pd

def cargar(ruta):
    """Lee un CSV (coma o punto y coma) o un Excel. Acepta el enlace Raw o el de la página de GitHub."""
    ruta = re.sub(r"^https://github\.com/([^/]+)/([^/]+)/blob/", r"https://raw.githubusercontent.com/\1/\2/", ruta.strip())
    if ruta.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(ruta)
    for codificacion in ("utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(ruta, sep=None, engine="python", encoding=codificacion)
        except UnicodeDecodeError:
            continue
        except pd.errors.ParserError:
            raise ValueError("No pude leerlo como tabla. Revisa que el enlace sea el de tu CSV (botón Raw).") from None
    raise ValueError("No pude leer el archivo: revisa que sea un CSV")

def sql(consulta):
    """Corre una consulta SQL sobre las tablas cargadas y muestra el resultado."""
    return duckdb.sql(consulta).df()

print("Listo. Sigue con la celda 2.")

Listo. Sigue con la celda 2.


## 2. Carga tu muestra

**Opción A (recomendada).** En GitHub abre tu CSV, copia la dirección de la página (o la del botón **Raw**) y pégala entre las comillas de `ENLACE`.

**Opción B.** Deja `ENLACE = ""` y corre la celda: aparece un botón para subir el archivo desde tu computador.

Con el enlace que viene (el caso de la papelería) deben salir **8 filas y 6 columnas**.

In [2]:
# Celda 2 · Cargar la muestra
ENLACE = "https://github.com/dmonsalvelondono8-cmyk/proyecto-datos-e7/blob/main/datos/originales/delitos_informaticos.csv"

if ENLACE:
    datos = cargar(ENLACE)
else:
    from google.colab import files
    subido = files.upload()
    datos = cargar(next(iter(subido)))

print(f"Cargué {len(datos)} filas y {len(datos.columns)} columnas.")
datos.head()

Cargué 537240 filas y 7 columnas.


,FECHA HECHO,COD_DEPTO,DEPARTAMENTO,COD_MUNI,MUNICIPIO,DESCRIPCION CONDUCTA,CANTIDAD
0,2026 Oct 31 12:00:00 AM,44,LA GUAJIRA,44035,ALBANIA,ARTICULO 269I. HURTO POR MEDIOS INFORMATICOS Y...,1
1,2026 Oct 31 12:00:00 AM,25,CUNDINAMARCA,25599,APULO,ARTICULO 269I. HURTO POR MEDIOS INFORMATICOS Y...,1
2,2026 Oct 31 12:00:00 AM,81,ARAUCA,81001,ARAUCA,ARTICULO 269F. VIOLACION DE DATOS PERSONALES,1
3,2026 Oct 31 12:00:00 AM,63,QUINDIO,63001,ARMENIA,ARTICULO 269I. HURTO POR MEDIOS INFORMATICOS Y...,1
4,2026 Oct 31 12:00:00 AM,8,ATLANTICO,8078,BARANOA,ARTICULO 269I. HURTO POR MEDIOS INFORMATICOS Y...,1


## 3. ¿Cuántas filas hay?

Compara con lo que escribiste en el diccionario, en «Filas en la muestra». Si no coincide, anótalo en la bitácora.

In [3]:
# Celda 3 · Contar filas
sql("""
SELECT COUNT(*) AS filas
FROM datos
""")

,filas
0,537240


## 4. Los nombres exactos de tus columnas

Cópialos de aquí. Si un nombre tiene espacios, tildes o mayúsculas, escríbelo entre **comillas dobles**: `"Fecha del hecho"`. Los valores de texto van entre **comillas simples**: `'completada'`.

In [4]:
# Celda 4 · Columnas de la muestra
list(datos.columns)

['FECHA HECHO',
 'COD_DEPTO',
 'DEPARTAMENTO',
 'COD_MUNI',
 'MUNICIPIO',
 'DESCRIPCION CONDUCTA',
 'CANTIDAD']

---
# Sesión 3 · Tus dos consultas

**Consulta 1 · Mirar y filtrar.** Primero cuenta todas las filas; después cuenta las que deja tu filtro; al final mira algunas. Cambia `estado`, `'completada'` y las columnas por las de tu muestra.

Con la papelería: 8 filas en total y 7 después del filtro.

In [5]:
# Celda 5 · Consulta 1a: filas antes del filtro
sql("""
SELECT COUNT(*) AS filas_total
FROM datos
""")

,filas_total
0,537240


In [16]:
# Identificar el código del municipio

sql("""
SELECT DISTINCT MUNICIPIO, COD_MUNI
FROM datos
WHERE MUNICIPIO = 'MEDELLIN'
""")

,MUNICIPIO,COD_MUNI
0,MEDELLIN,5001


In [17]:
# Pregunta: ¿Cuántas filas corresponden a Medellín?
# Datos: datos/originales/delitos_informaticos.csv (537.240 filas)
# Cifra de control: 42.367 filas corresponden a Medellín

sql("""
SELECT COUNT(*) AS filas_medellin
FROM datos
WHERE COD_MUNI = 5001;
""")

,filas_medellin
0,42367


In [18]:
# Pregunta: ¿Cuántos casos de delitos informáticos se registraron por año en Medellín?
# Datos: datos/originales/delitos_informaticos.csv (537.240 filas)
# Cifra de control: 5.711 casos en 2025

sql("""
SELECT LEFT("FECHA HECHO", 4) AS anio,
       SUM(CANTIDAD) AS total_casos
FROM datos
WHERE COD_MUNI = 5001
GROUP BY anio
ORDER BY anio;
""")

,anio,total_casos
0,2009,14.0
1,2010,8.0
2,2011,5.0
3,2012,22.0
4,2013,50.0
5,2014,8.0
6,2015,807.0
7,2016,963.0
8,2017,1472.0
9,2018,1686.0


In [9]:
# Celda 6 · Consulta 1b: filas después del filtro
# Consulta 1 · Los cinco delitos con más casos

sql("""
SELECT "DESCRIPCION CONDUCTA" AS delito,
       SUM(CANTIDAD) AS total_casos
FROM datos
WHERE COD_MUNI = 5001
GROUP BY "DESCRIPCION CONDUCTA"
ORDER BY total_casos DESC
LIMIT 5
""")

,delito,total_casos
0,ARTICULO 269I. HURTO POR MEDIOS INFORMATICOS Y...,20513.0
1,ARTICULO 269F. VIOLACION DE DATOS PERSONALES,9337.0
2,ARTICULO 269A. ACCESO ABUSIVO A UN SISTEMA INF...,7530.0
3,ARTICULO 269G. SUPLANTACION DE SITIOS WEB PARA...,2333.0
4,ARTICULO 269J. TRANSFERENCIA NO CONSENTIDA DE ...,1556.0


In [11]:
# Consulta 2 · Casos por mes y tipo de delito en Medellín
sql("""
SELECT STRFTIME(STRPTIME("FECHA HECHO", '%Y %b %d %I:%M:%S %p'), '%Y-%m') AS mes,
       "DESCRIPCION CONDUCTA" AS elito,
       SUM(dCANTIDAD) AS total_casos
FROM datos
WHERE COD_MUNI = 5001
GROUP BY mes, delito
ORDER BY mes, delito
LIMIT 10
""")

,mes,delito,total_casos
0,2009-01,ARTICULO 269I. HURTO POR MEDIOS INFORMATICOS Y...,1.0
1,2009-03,ARTICULO 269I. HURTO POR MEDIOS INFORMATICOS Y...,1.0
2,2009-04,ARTICULO 269I. HURTO POR MEDIOS INFORMATICOS Y...,3.0
3,2009-05,ARTICULO 269A. ACCESO ABUSIVO A UN SISTEMA INF...,1.0
4,2009-07,ARTICULO 269A. ACCESO ABUSIVO A UN SISTEMA INF...,1.0
5,2009-10,ARTICULO 269A. ACCESO ABUSIVO A UN SISTEMA INF...,2.0
6,2009-10,ARTICULO 269C. INTERCEPTACION DE DATOS INFORMA...,1.0
7,2009-10,ARTICULO 269I. HURTO POR MEDIOS INFORMATICOS Y...,1.0
8,2009-11,ARTICULO 269I. HURTO POR MEDIOS INFORMATICOS Y...,1.0
9,2009-12,ARTICULO 269A. ACCESO ABUSIVO A UN SISTEMA INF...,1.0


**Consulta 2 · Resumir.** Cuenta cuántas filas hay en cada grupo. Cambia `estado` por la columna con la que quieres comparar (comuna, mes, categoría…).

**Cifra de control:** la suma de los grupos debe dar lo mismo que el total de la celda 5. Con la papelería: completada 7 + cancelada 1 = 8.

In [13]:
# Celda 8 · Consulta 2: resumen por grupo
# consulta 3: los departamentos que registran más casos

sql("""
SELECT DEPARTAMENTO,
       SUM(CANTIDAD) AS total_casos
FROM datos
GROUP BY DEPARTAMENTO
ORDER BY total_casos DESC
LIMIT 10
""")

,DEPARTAMENTO,total_casos
0,BOGOTA D.C.,168048.0
1,ANTIOQUIA,72753.0
2,VALLE DEL CAUCA,52599.0
3,CUNDINAMARCA,32601.0
4,ATLANTICO,25569.0
5,SANTANDER,24949.0
6,TOLIMA,15025.0
7,BOLIVAR,14160.0
8,META,12565.0
9,BOYACA,12333.0


In [15]:
# Celda 9 · Cifra de control: ¿la suma de los grupos da el total?
# Consulta 4: el cambio del total de casos en Medellín por año

sql("""
SELECT LEFT("FECHA HECHO", 4) AS anio,
       SUM(CANTIDAD) AS total_casos
FROM datos
WHERE COD_MUNI = 5001
GROUP BY anio
ORDER BY anio
""")

,anio,total_casos
0,2009,14.0
1,2010,8.0
2,2011,5.0
3,2012,22.0
4,2013,50.0
5,2014,8.0
6,2015,807.0
7,2016,963.0
8,2017,1472.0
9,2018,1686.0


## Guarda tus consultas en GitHub

En tu repositorio, entra a `consultas/`, **Add file › Create new file**, ponle nombre (`01_mirar.sql`, `02_resumen.sql`), pega la consulta con tres comentarios arriba y haz commit con un mensaje que diga qué averiguaste:

```sql
-- Pregunta: ¿cuántas ventas se completaron?
-- Datos: datos/originales/ventas.csv (8 filas)
-- Cifra de control: 8 en total, 7 completadas
SELECT COUNT(*) AS filas_filtradas
FROM datos
WHERE estado = 'completada';
```

Mira el ejemplo resuelto en `consultas/` del repositorio de la papelería.

---
¿Algo falló? Regla de 3 y, si sigues bloqueado, un ticket en la **Cola de ayuda** con el mensaje de error copiado. Nunca pegues contraseñas ni datos personales.